# RNA-Seq Transcript Overlap Analysis (Gallus gallus)

**Author:** Hera Dashnyam  

This notebook analyzes RNA-Seq assembled transcripts and compares them with the reference genome annotation to identify novel genes and transcript isoforms in the chicken genome.

---

## Objective

The goal of this analysis is to determine how transcripts reconstructed from RNA-Seq assembly relate to known reference gene annotations.

Specifically, this notebook:

• parses the reference gene annotation and assembled transcriptome  
• normalizes gene and transcript identifiers  
• performs genomic interval intersections using **pybedtools**  
• classifies transcripts based on overlap with reference genes  
• generates gene-level summaries and visualization outputs  
• exports IGV tracks for manual inspection of candidate loci

---

## Transcript Categories

Assembled transcripts are classified into three major categories based on their overlap with reference gene annotations:

**C1 — Novel gene / novel isoforms**  
A transcript locus located in an intergenic region with no overlap with any reference gene.

**C2 — Novel gene overlapping reference isoforms**  
A transcript locus that overlaps existing gene annotations but represents a distinct transcriptional unit.

**C3 — Known gene with novel isoforms**  
A previously annotated gene where RNA-Seq assembly reveals additional transcript isoforms.

---

## Output Files

The pipeline produces several tables and visualization tracks:

**Gene-level tables**

- `reference_genes.csv`
- `stringtie_gene_summary.csv`

**Transcript-level table**

- `assembled_transcripts.csv`

**Comparison table**

- `gene_comparison_unified.csv`

**IGV visualization tracks**

- `loci_C1.bed`
- `loci_C2.bed`
- `loci_C3.bed`

---

## Notes

• Overlap analysis is performed using genomic interval intersections via **pybedtools**.  
• Transcript overlap categories are derived from intersection relationships with reference genes.  
• Gene-level categories (C1/C2/C3) summarize transcript classifications within each StringTie locus.  
• A dominant strand is assigned to each locus based on the majority transcript strand.

---

## Data

Reference genome annotation  
bGalGal1.mat.broiler.GRCg7b.annotation.gtf

RNA-Seq transcriptome assembly  
YapPool_merge.gtf

# setup and paths

In [ ]:
# System and packages
!apt-get update -qq && apt-get install -y -qq bedtools
!pip install --quiet pybedtools==0.9.1 gffpandas==1.2.0

import re, csv
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import pybedtools

# Drive
from google.colab import drive
drive.mount('/content/drive')

# Base folder
BASE = Path('/content/drive/MyDrive/HeraDashnyam')

# Prefer sorted StringTie if present
st_candidates = [
    BASE / 'YapPool_merged.sorted.gtf',
    BASE / 'YapPool_merged.gtf',
]
assembled_gtf_path = next((p for p in st_candidates if p.exists()), None)

# Reference preference: GFF3 then GTF
ref_candidates = [
    BASE / 'Gallus_gallus.bGalGal1.mat.broiler.GRCg7b.113.gff3',
    BASE / 'bGalGal1.mat.broiler.GRCg7b.annotation.gtf',
]
reference_path = next((p for p in ref_candidates if p.exists()), None)

assert assembled_gtf_path is not None, "No StringTie merged GTF found. Put YapPool_merged.sorted.gtf or YapPool_merged.gtf in HeraDashnyam."
assert reference_path is not None, "No reference annotation found. Place GFF3 or GTF in HeraDashnyam."

# Output directory
OUT = BASE / 'ResTask03_outputs'
OUT.mkdir(parents=True, exist_ok=True)

# Controls
STRAND_AWARE = False    # default strand agnostic
print("Using StringTie:", assembled_gtf_path.name)
print("Using reference:", reference_path.name)
print("Outputs to    :", str(OUT))


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using StringTie: YapPool_merged.sorted.gtf
Using reference: Gallus_gallus.bGalGal1.mat.broiler.GRCg7b.113.gff3
Outputs to    : /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs


## Helpers

In [ ]:
def norm_id(x: str):
    if pd.isna(x):
        return x
    return re.sub(r'^(gene:|transcript:)', '', str(x))

def bed_from_df(df, chrom='seq_id', start='start', end='end', name='name', score='score', strand='strand'):
    t = df[[chrom, start, end, name, score, strand]].copy()
    t[chrom] = t[chrom].astype(str)
    t[start] = t[start].astype(int) - 1           # BED zero based
    t[end]   = t[end].astype(int)                 # BED end exclusive
    t[name]  = t[name].fillna('.').astype(str)
    t[score] = t[score].fillna('0').astype(str)
    t[strand]= t[strand].fillna('.').astype(str)
    return t

def rank_within_group(df, group_col, value_col, asc=True, rank_col='rank_in_gene'):
    df[rank_col] = df.groupby(group_col)[value_col].rank(method='dense', ascending=asc).astype(int)
    return df

def dominant_strand(series):
    vals = [s for s in series if s in ['+','-']]
    c = Counter(vals)
    if c.get('+',0) > c.get('-',0): return '+'
    if c.get('-',0) > c.get('+',0): return '-'
    return '.'

def extract_attr(text, key):
    # Works for GTF or GFF3 style attributes
    s = str(text)
    m = re.search(rf'{key}\s*[= ]\s*"?([^";]+)"?', s)
    return m.group(1) if m else None


## Parse reference to gene and transcript tables

In [ ]:
def parse_reference_to_tables(reference_path: Path):
    # Read generic nine column GFF GTF
    ref = pd.read_csv(reference_path, sep='\t', comment='#', header=None,
                      names=['seq_id','source','type','start','end','score','strand','phase','attributes'])

    # Keep gene and transcript like features
    ref = ref[ref['type'].isin(['gene','mRNA','transcript'])].copy()

    # IDs and names
    ref['gene_id']       = ref['attributes'].apply(lambda a: extract_attr(a, 'gene_id') or extract_attr(a, 'ID'))
    ref['gene_name']     = ref['attributes'].apply(lambda a: extract_attr(a, 'gene_name'))
    # Transcript id: prefer transcript_id, otherwise use ID when Parent exists to avoid gene IDs
    ref['transcript_id'] = ref['attributes'].apply(
        lambda a: extract_attr(a, 'transcript_id') or (extract_attr(a, 'ID') if extract_attr(a, 'Parent') else None)
    )

    ref['gene_id']       = ref['gene_id'].map(norm_id)
    ref['transcript_id'] = ref['transcript_id'].map(norm_id)

    # Gene table
    reference_genes = (ref[ref['type']=='gene'][['seq_id','start','end','strand','gene_id','gene_name']]
                       .drop_duplicates().copy())
    reference_genes['source']   = 'reference'
    reference_genes['category'] = 'REF'
    reference_genes['length']   = reference_genes['end'].astype(int) - reference_genes['start'].astype(int) + 1

    # Transcript table
    reference_tx = (ref[ref['type'].isin(['mRNA','transcript'])]
                    [['seq_id','start','end','strand','transcript_id','gene_id','gene_name']]
                    .drop_duplicates().copy())
    reference_tx['source'] = 'reference'
    reference_tx['length'] = reference_tx['end'].astype(int) - reference_tx['start'].astype(int) + 1

    return reference_genes, reference_tx

reference_genes, reference_tx = parse_reference_to_tables(reference_path)

# Export 1
ref_genes_out = OUT / 'reference_genes.csv'
reference_genes.to_csv(ref_genes_out, index=False)
print("Wrote:", ref_genes_out)


/tmp/ipython-input-3270544123.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  ref = pd.read_csv(reference_path, sep='\t', comment='#', header=None,


Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/reference_genes.csv


## Parse StringTie GTF, add exon rollup, compute transcript ranks, NOVEL or OVERLAP

In [ ]:
import gffpandas.gffpandas as gffpd

def parse_stringtie_gtf(gtf_file: Path):
    df = gffpd.read_gff3(str(gtf_file)).df
    df = df[df['type'].isin(['transcript','exon'])].copy()

    def A(col, key):
        return col.str.extract(key + r'\s*"?(?P<v>[^";]+)"?', expand=False)

    df['gene_id']        = A(df['attributes'], 'gene_id').map(norm_id)
    df['transcript_id']  = A(df['attributes'], 'transcript_id').map(norm_id)
    df['reference_id']   = A(df['attributes'], 'reference_id').map(norm_id)
    df['ref_gene_id']    = A(df['attributes'], 'ref_gene_id').map(norm_id)
    df['ref_gene_name']  = A(df['attributes'], 'ref_gene_name')
    df['cov']            = pd.to_numeric(A(df['attributes'], 'cov'), errors='coerce')
    df['FPKM']           = pd.to_numeric(A(df['attributes'], 'FPKM'), errors='coerce')
    df['TPM']            = pd.to_numeric(A(df['attributes'], 'TPM'), errors='coerce')
    df['exon_number']    = pd.to_numeric(A(df['attributes'], 'exon_number'), errors='coerce')

    return df

st = parse_stringtie_gtf(assembled_gtf_path)
tx = st[st['type']=='transcript'].copy()
ex = st[st['type']=='exon'].copy()

def exon_rollup(ex_df):
    rows = []
    for tid, grp in ex_df.groupby('transcript_id'):
        pairs = [f"{int(s)}-{int(e)}" for s, e in zip(grp['start'], grp['end'])]
        total_len = (grp['end'].astype(int) - grp['start'].astype(int)).sum()
        rows.append((tid, len(grp), ", ".join(pairs), int(total_len)))
    return pd.DataFrame(rows, columns=['transcript_id','exon_count','exon_details','total_exon_length'])

ex_info = exon_rollup(ex)
tx = tx.merge(ex_info, on='transcript_id', how='left')

# Transcript length and ranks inside gene short to long
tx['transcript_length'] = tx['end'].astype(int) - tx['start'].astype(int) + 1
tx = rank_within_group(tx, 'gene_id', 'transcript_length', asc=True, rank_col='rank_in_gene')

# Transcript NOVEL or OVERLAP by intersect with reference transcripts
ref_tx_bed = bed_from_df(reference_tx.assign(name=reference_tx['transcript_id'], score='0'),
                         name='transcript_id', score='source')
st_tx_bed  = bed_from_df(tx.assign(name=tx['transcript_id'], score='0'),
                         name='transcript_id', score='source')

A = pybedtools.BedTool.from_dataframe(st_tx_bed)
B = pybedtools.BedTool.from_dataframe(ref_tx_bed)

if STRAND_AWARE:
    inter_t = A.intersect(B, s=True, wa=True, wb=True)
else:
    inter_t = A.intersect(B, s=False, wa=True, wb=True)

overlap_tids = set(r[3] for r in inter_t)   # column 4 of A block

tx['category'] = np.where(tx['transcript_id'].isin(overlap_tids), 'OVERLAP', 'NOVEL')

assembled_transcripts = tx[['seq_id','start','end','strand',
                            'gene_id','transcript_id','reference_id','ref_gene_id','ref_gene_name',
                            'cov','FPKM','TPM','exon_count','exon_details','total_exon_length',
                            'transcript_length','rank_in_gene','category']].copy()
assembled_transcripts['source'] = 'stringtie'

# Export 2
assembled_transcripts_out = OUT / 'assembled_transcripts.csv'
assembled_transcripts.to_csv(assembled_transcripts_out, index=False)
print("Wrote:", assembled_transcripts_out)


/usr/local/lib/python3.12/dist-packages/gffpandas/gffpandas.py:32: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_table(self._gff_file, comment='#',


Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/assembled_transcripts.csv


# Gene level rollup and C1 C2 C3



In [ ]:
# Gene locus rollup from StringTie transcripts
g = tx.groupby('gene_id').agg(
    seq_id=('seq_id','first'),
    start =('start','min'),
    end   =('end','max'),
    strands=('strand', list),
    transcript_ids=('transcript_id', list),
    reference_ids =('reference_id', lambda s: sorted(set([x for x in s if pd.notna(x)]))),
    ref_gene_ids  =('ref_gene_id',  lambda s: sorted(set([x for x in s if pd.notna(x)]))),
    ref_gene_names=('ref_gene_name',lambda s: sorted(set([x for x in s if pd.notna(x)]))),
    transcript_count=('transcript_id','count'),
    overlap_tx_count=('category', lambda s: int((pd.Series(s)=='OVERLAP').sum()))
).reset_index()

g['strand'] = g['strands'].apply(dominant_strand)
g.drop(columns=['strands'], inplace=True)
g['length'] = g['end'].astype(int) - g['start'].astype(int) + 1

# Overlap of StringTie gene loci with reference genes
ref_gene_bed = bed_from_df(reference_genes.assign(name=reference_genes['gene_id'], score='0'),
                           name='gene_id', score='source')
st_gene_bed  = bed_from_df(g.assign(name=g['gene_id'], score='0'),
                           name='gene_id', score='strand')

GA = pybedtools.BedTool.from_dataframe(st_gene_bed)
GB = pybedtools.BedTool.from_dataframe(ref_gene_bed)

if STRAND_AWARE:
    inter_g = GA.intersect(GB, s=True, wa=True, wb=True)
else:
    inter_g = GA.intersect(GB, s=False, wa=True, wb=True)

st_to_ref = defaultdict(set)
for r in inter_g:
    st_to_ref[r[3]].add(r[9])

g['overlap_ref_genes'] = g['gene_id'].map(lambda x: sorted(st_to_ref.get(x, set())))

# Category logic
def classify_c123(row):
    overlaps = len(row['overlap_ref_genes']) > 0
    has_identity = (len(row['ref_gene_ids']) > 0) or (len(row['ref_gene_names']) > 0)
    if not overlaps:
        return 'C1'  # novel locus no overlap
    if not has_identity:
        return 'C2'  # overlaps but no identity linkage
    return 'C3'      # overlaps and has identity linkage

g['C_category'] = g.apply(classify_c123, axis=1)

stringtie_gene_summary = g[['seq_id','start','end','strand','gene_id',
                            'length','transcript_count','transcript_ids',
                            'ref_gene_ids','ref_gene_names','overlap_ref_genes',
                            'overlap_tx_count','C_category']].copy()
stringtie_gene_summary['source'] = 'stringtie'

# Export 3
st_gene_summary_out = OUT / 'stringtie_gene_summary.csv'
stringtie_gene_summary.to_csv(st_gene_summary_out, index=False)
print("Wrote:", st_gene_summary_out)


Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/stringtie_gene_summary.csv


## Unified comparison table with the same schema



In [ ]:
# Canonical schema
cols = ['source','level','seq_id','start','end','strand','length',
        'gene_id','gene_name','transcript_id','transcript_length',
        'rank_in_gene','exon_count','total_exon_length','category',
        'ref_gene_id','ref_gene_name','reference_id','C_category']

ref_parts = []
st_parts  = []

# Reference genes
rg = reference_genes.copy()
rg['level'] = 'gene'
rg['transcript_id'] = np.nan
rg['transcript_length'] = np.nan
rg['rank_in_gene'] = np.nan
rg['exon_count'] = np.nan
rg['total_exon_length'] = np.nan
rg['ref_gene_id'] = rg['gene_id']
rg['ref_gene_name'] = rg['gene_name']
rg['reference_id'] = np.nan
rg['C_category'] = 'REF'
ref_parts.append(rg.rename(columns={'category':'category'})[cols])

# Reference transcripts
rt = reference_tx.copy()
rt['level'] = 'transcript'
rt['length'] = rt['length']
rt['rank_in_gene'] = np.nan
rt['exon_count'] = np.nan
rt['total_exon_length'] = np.nan
rt['ref_gene_id'] = rt['gene_id']
rt['ref_gene_name'] = rt['gene_name']
rt['reference_id'] = rt['transcript_id']
rt['category'] = 'REF_TX'
rt['C_category'] = 'REF'
ref_parts.append(rt.assign(transcript_length=rt['length'])[cols])

ref_part = pd.concat(ref_parts, ignore_index=True)

# StringTie genes
sg = stringtie_gene_summary.copy()
sg['level'] = 'gene'
sg['gene_name'] = np.nan
sg['transcript_id'] = np.nan
sg['transcript_length'] = np.nan
sg['rank_in_gene'] = np.nan
sg['exon_count'] = np.nan
sg['total_exon_length'] = np.nan
sg['ref_gene_id'] = sg['ref_gene_ids'].apply(lambda x: x[0] if len(x)>0 else np.nan)
sg['ref_gene_name'] = sg['ref_gene_names'].apply(lambda x: x[0] if len(x)>0 else np.nan)
sg['reference_id'] = np.nan
sg['category'] = sg['C_category']
st_parts.append(sg[cols])

# StringTie transcripts
stt = assembled_transcripts.copy()
stt['level'] = 'transcript'
stt['gene_name'] = stt['ref_gene_name']    # best symbol if present
stt['length'] = stt['transcript_length']
stt['C_category'] = np.nan
st_parts.append(stt[cols])

st_part = pd.concat(st_parts, ignore_index=True)

gene_comparison_unified = pd.concat([ref_part, st_part], ignore_index=True)

# Export 4
unified_out = OUT / 'gene_comparison_unified.csv'
gene_comparison_unified.to_csv(unified_out, index=False)
print("Wrote:", unified_out)


Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/gene_comparison_unified.csv


# BED loci per category for IGV

In [ ]:
def write_bed(df, path):
    bed = bed_from_df(
        df.assign(name=df['gene_id'], score=df['length'])
          [['seq_id','start','end','name','score','strand']],
        name='name', score='score'
    )
    bed.to_csv(path, sep='\t', header=False, index=False)

for cat in ['C1','C2','C3']:
    loci = stringtie_gene_summary[stringtie_gene_summary['C_category']==cat]
    outp = OUT / f"loci_{cat}.bed"
    write_bed(loci, outp)
    print("Wrote:", outp)


Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/loci_C1.bed
Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/loci_C2.bed
Wrote: /content/drive/MyDrive/HeraDashnyam/ResTask03_outputs/loci_C3.bed


# Sanity report and quick QA

In [ ]:
def nfmt(n):
    return f"{int(n):,}"

# Transcript level
t_counts = assembled_transcripts['category'].value_counts(dropna=False)
print("\nTranscript categories")
print(t_counts)

# Gene level
g_counts = stringtie_gene_summary['C_category'].value_counts(dropna=False)
print("\nGene categories C1 C2 C3")
print(g_counts)

# Top genes by number of NOVEL transcripts
novel_tx = assembled_transcripts[assembled_transcripts['category']=='NOVEL']
novel_per_gene = novel_tx.groupby('gene_id').size().sort_values(ascending=False).head(10)
print("\nTop genes by number of NOVEL transcripts")
print(novel_per_gene)

# Dominant strand tie count
tie_count = (stringtie_gene_summary['strand']=='.').sum()
print("\nGene loci with tie strand equals dot:", tie_count)

# Column checks
print("\nreference_genes.csv columns:", list(reference_genes.columns))
print("assembled_transcripts.csv columns:", list(assembled_transcripts.columns))
print("stringtie_gene_summary.csv columns:", list(stringtie_gene_summary.columns))
print("gene_comparison_unified.csv columns:", list(gene_comparison_unified.columns))

# File list
print("\nFiles in OUT:")
for p in sorted(OUT.glob('*')):
    print("  ", p.name)



Transcript categories
category
OVERLAP    71701
NOVEL      28916
Name: count, dtype: int64

Gene categories C1 C2 C3
C_category
C3    16220
C1    11959
C2      606
Name: count, dtype: int64

Top genes by number of NOVEL transcripts
gene_id
MSTRG.14856    95
MSTRG.19017    51
MSTRG.19014    51
MSTRG.18766    36
MSTRG.16297    29
MSTRG.19012    28
MSTRG.2        26
MSTRG.13263    24
MSTRG.12795    22
MSTRG.8668     22
dtype: int64

Gene loci with tie strand equals dot: 259

reference_genes.csv columns: ['seq_id', 'start', 'end', 'strand', 'gene_id', 'gene_name', 'source', 'category', 'length']
assembled_transcripts.csv columns: ['seq_id', 'start', 'end', 'strand', 'gene_id', 'transcript_id', 'reference_id', 'ref_gene_id', 'ref_gene_name', 'cov', 'FPKM', 'TPM', 'exon_count', 'exon_details', 'total_exon_length', 'transcript_length', 'rank_in_gene', 'category', 'source']
stringtie_gene_summary.csv columns: ['seq_id', 'start', 'end', 'strand', 'gene_id', 'length', 'transcript_count', 'trans

# chromosome name harmonization

In [ ]:
# Enable only if needed
ENABLE_CHR_MAP = False

# Example mapping pattern
CHR_MAP = {
    'MT':'MT',
    'W':'W',
    'Z':'Z',
    # '1':'1', '2':'2', etc. add only if you need transformations
}

def apply_chr_map(df, col='seq_id', m=CHR_MAP):
    if not ENABLE_CHR_MAP:
        return df
    d = df.copy()
    d[col] = d[col].astype(str).map(lambda x: m.get(x, x))
    return d

# If you enable, run these before any bed_from_df call:
# reference_genes = apply_chr_map(reference_genes)
# reference_tx    = apply_chr_map(reference_tx)
# tx              = apply_chr_map(tx)
# g               = apply_chr_map(g)


# What each output contains

**reference_genes.csv**

one row per reference gene with seq id, start, end, strand, gene id, gene name, length and source equals reference

**assembled_transcripts.csv**

one row per StringTie transcript with gene and transcript ids normalized, exon counts and details, total exon length, transcript length, rank in gene, category equals NOVEL or OVERLAP, plus reference identity fields if present

**stringtie_gene_summary.csv**

one row per StringTie gene locus with dominant strand, overlap reference gene list, counts of overlapping transcripts, and C category

**gene_comparison_unified.csv**

stacked table for reference and StringTie with identical schema for quick diff and filters

**loci_C1.bed, loci_C2.bed, loci_C3.bed**

gene loci per category for IGV or bedtools follow ups